In [4]:
import os

print(os.getcwd())
if not os.getcwd().endswith("app"):
    os.chdir("../app")
    print(os.getcwd())

%load_ext autoreload
%autoreload 2

from src.config import Configuration
CONFIG = Configuration()

/home/turbotowerlnx/Documents/Master/ALC/ALC-Lab-Final/app
The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# Extract embeddings

In [ ]:
import torch
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor
from qwen_vl_utils import process_vision_info
import subprocess
import tempfile
import glob
from PIL import Image

# 1. Load Model (Using 2B version to save VRAM. Use 7B if you have 24GB+ VRAM)
model_id = "Qwen/Qwen2-VL-2B-Instruct"
model = Qwen2VLForConditionalGeneration.from_pretrained(
    model_id, 
    device_map="cuda", 
    torch_dtype=torch.float16
).eval()

processor = AutoProcessor.from_pretrained(model_id)



def extract_frames_with_ffmpeg(video_path, fps=1.0, max_frames=64):
    """
    Use ffmpeg to extract frames, ignoring decode errors.
    Returns a list of PIL Images, or raises if nothing could be decoded.
    """
    with tempfile.TemporaryDirectory() as tmpdir:
        out_pattern = os.path.join(tmpdir, "frame_%04d.jpg")
        cmd = [
            "ffmpeg",
            "-err_detect", "ignore_err",   # keep going past bitstream errors
            "-i", video_path,
            "-vf", f"fps={fps}",
            "-q:v", "2",                   # jpeg quality
            "-frames:v", str(max_frames),  # cap frame count
            out_pattern,
            "-y", "-loglevel", "error"     # suppress ffmpeg noise
        ]
        result = subprocess.run(cmd, capture_output=True)
        
        frame_files = sorted(glob.glob(os.path.join(tmpdir, "frame_*.jpg")))
        if not frame_files:
            raise RuntimeError(f"ffmpeg extracted 0 frames from: {video_path}")
        
        frames = [Image.open(f).copy() for f in frame_files]  # .copy() before tmpdir cleanup
    
    return frames


def get_video_embedding(video_path):
    messages = [
        {"role": "user", "content": [
            {"type": "video", "video": video_path, "max_pixels": 360 * 360, "fps": 1.0},
        ]}
    ]
    
    # --- Primary path: normal decoding ---
    try:
        image_inputs, video_inputs = process_vision_info(messages)
        used_fallback = False
    except Exception:
        # --- Fallback: extract salvageable frames via ffmpeg ---
        try:
            frames = extract_frames_with_ffmpeg(video_path, fps=1.0)
            used_fallback = True
        except Exception as e:
            raise RuntimeError(f"Could not decode video: {video_path}") from e

        # Feed recovered frames as individual images instead of a video tensor
        fallback_messages = [
            {"role": "user", "content": [
                {"type": "image", "image": frame} for frame in frames
            ]}
        ]
        image_inputs, video_inputs = process_vision_info(fallback_messages)

    inputs = processor(
        text=[""],
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt"
    ).to("cuda")

    with torch.no_grad():
        if used_fallback:
            # Frames came in as images, not video tensors
            visual_features = model.model.visual(
                inputs.pixel_values,
                grid_thw=inputs.image_grid_thw
            )
        else:
            visual_features = model.model.visual(
                inputs.pixel_values_videos,
                grid_thw=inputs.video_grid_thw
            )
        
        token_features = (
            visual_features.last_hidden_state
            if hasattr(visual_features, "last_hidden_state")
            else visual_features
        )
        video_embedding = token_features.mean(dim=0)

    return video_embedding.cpu().numpy(), used_fallback

Loading weights: 100%|██████████| 729/729 [00:00<00:00, 1472.86it/s]


In [7]:
import pickle
from tqdm import tqdm
from maikol_utils.file_utils import list_dir_files

# --- Main loop (updated to track fallbacks separately) ---
videos, n = list_dir_files(CONFIG.videos_path)
embeddings_dict = {}
failed_videos = []
fallback_videos = []  # recovered but partial

for video in tqdm(videos):
    try:
        vector, was_fallback = get_video_embedding(video)
        embeddings_dict[os.path.basename(video)] = vector
        if was_fallback:
            fallback_videos.append(video)
    except Exception as e:
        failed_videos.append((video, str(e)))
        print(f"Skipped: {video} -> {e}")

print(f"✓ Embedded: {len(embeddings_dict)} | ⚠ Partial (recovered): {len(fallback_videos)} | ✗ Failed: {len(failed_videos)}")

with open(os.path.join(CONFIG.full_dataset_path, "video_embeddings.pkl"), "wb") as f:
    pickle.dump(embeddings_dict, f)

  0%|          | 3/2524 [00:00<12:38,  3.32it/s][mov,mp4,m4a,3gp,3g2,mj2 @ 0x615cc26e7840] stream 1, offset 0xc0186: partial file
video_reader_backend decord error, use torchvision as default, msg: [22:35:58] /github/workspace/src/video/video_reader.cc:486: Error: av_read_frame failed with 1094995529
  5%|▍         | 120/2524 [00:50<12:23,  3.23it/s][h264 @ 0x615c9b606040] Invalid NAL unit size (16738 > 4736).
[h264 @ 0x615c9b606040] Error splitting the input into NAL units.
video_reader_backend decord error, use torchvision as default, msg: [22:36:48] /github/workspace/src/video/ffmpeg/threaded_decoder.cc:292: [22:36:48] /github/workspace/src/video/ffmpeg/filter_graph.cc:100: Check failed: av_buffersrc_add_frame_flags(buffersrc_ctx_, frame, AV_BUFFERSRC_FLAG_KEEP_REF) >= 0 (-22 vs. 0) Error while feeding the filter graph
  6%|▋         | 160/2524 [01:10<28:00,  1.41it/s][mov,mp4,m4a,3gp,3g2,mj2 @ 0x615cc2ccda80] stream 0, offset 0x7995b: partial file
video_reader_backend decord error,

✓ Embedded: 2524 | ⚠ Partial (recovered): 26 | ✗ Failed: 0


In [8]:
embeddings_dict

{'6567557435626622214.mp4': array([ 2.398, -1.084,  3.23 , ...,  4.547,  2.436,  1.504],
       shape=(1280,), dtype=float16),
 '6605282293726579974.mp4': array([ 3.191 , -0.4075,  2.143 , ...,  3.102 ,  1.903 ,  0.964 ],
       shape=(1280,), dtype=float16),
 '6630477888766348549.mp4': array([2.562 , 0.4783, 2.898 , ..., 2.822 , 1.783 , 1.27  ],
       shape=(1280,), dtype=float16),
 '6657214348496211205.mp4': array([ 3.104 , -0.1233,  2.283 , ...,  3.082 ,  2.617 ,  2.41  ],
       shape=(1280,), dtype=float16),
 '6683497456589606149.mp4': array([ 2.23  , -0.4531,  3.088 , ...,  3.355 ,  1.936 ,  1.366 ],
       shape=(1280,), dtype=float16),
 '6717331892019989766.mp4': array([2.768 , 0.1816, 3.322 , ..., 2.709 , 2.057 , 1.715 ],
       shape=(1280,), dtype=float16),
 '6724072000509185286.mp4': array([ 2.443, -0.193,  2.87 , ...,  2.838,  1.323,  1.852],
       shape=(1280,), dtype=float16),
 '6739312750398409990.mp4': array([2.467 , 0.1043, 3.467 , ..., 3.775 , 2.182 , 0.52  ],
    

In [9]:
with open(os.path.join(CONFIG.full_dataset_path, "video_embeddings.pkl"), "rb") as f:
    embeddings_dict_ = pickle.load(f)

In [10]:
embeddings_dict_

{'6567557435626622214.mp4': array([ 2.398, -1.084,  3.23 , ...,  4.547,  2.436,  1.504],
       shape=(1280,), dtype=float16),
 '6605282293726579974.mp4': array([ 3.191 , -0.4075,  2.143 , ...,  3.102 ,  1.903 ,  0.964 ],
       shape=(1280,), dtype=float16),
 '6630477888766348549.mp4': array([2.562 , 0.4783, 2.898 , ..., 2.822 , 1.783 , 1.27  ],
       shape=(1280,), dtype=float16),
 '6657214348496211205.mp4': array([ 3.104 , -0.1233,  2.283 , ...,  3.082 ,  2.617 ,  2.41  ],
       shape=(1280,), dtype=float16),
 '6683497456589606149.mp4': array([ 2.23  , -0.4531,  3.088 , ...,  3.355 ,  1.936 ,  1.366 ],
       shape=(1280,), dtype=float16),
 '6717331892019989766.mp4': array([2.768 , 0.1816, 3.322 , ..., 2.709 , 2.057 , 1.715 ],
       shape=(1280,), dtype=float16),
 '6724072000509185286.mp4': array([ 2.443, -0.193,  2.87 , ...,  2.838,  1.323,  1.852],
       shape=(1280,), dtype=float16),
 '6739312750398409990.mp4': array([2.467 , 0.1043, 3.467 , ..., 3.775 , 2.182 , 0.52  ],
    